# E4 — Decomposition sweep (the paper's Table 1) (v1, 2026-07-11)

Implements plan §E4 (`research/EXPERIMENT_PLAN_2026-07-08.md`): for every tuned (dataset, ε) cell from E1,
audits the SAME 3 canonical-seed models with the passive Raw-LiRA (K=32 shadows, 5120 disjoint queries) and
matched-canary (1000 canaries, 5 audit seeds) tracks, both estimators (holdout Wilson primary / GDP), then
attributes the RDP-vs-empirical gap: accounting / threat-model / estimator / residual shares that telescope
to 1 (gate G2 asserts this per cell).

**Colab edition** — upload `decomposition_bundle_v1.zip` when prompted, then the `hpo_e1e2_results_<dataset>.zip` produced by the E1+E2 notebook.

**Prerequisites:** E1 best configs (required). E2 checkpoints (optional — targets retrain
automatically if absent, which also regenerates the E2 artifacts). Run E3 (saturation v4) before quoting
residual shares as attributable: if E3 says STILL CLIMBING, report "residual ≥ partly weak-auditor" (§0).

**Batching:** one (dataset, ε) cell per session (~1–2 h on T4, dominated by 32 shadows + 15 canary retrains).
Results flush after every row; re-running skips finished rows. Canary track is MNIST-only for now
(no tabular canary design exists — diabetes cells carry `threat_share_excluded=true`).

**Gate G3 (once, before the first real cell):** run `SMOKE=1 python codex/run_decomposition_sweep.py
--dataset mnist --epsilon 1.0` — every row must come out `quality_flag="exploratory"` (the pipeline must
refuse to certify from tiny budgets). Smoke writes to a separate results root.

Bundle sha256: `9dcf77389d176661e68308e0c069acac4e0e30f1e33f1cc85f54bc94d313f06d`

In [ ]:
from google.colab import files
up = files.upload()  # choose decomposition_bundle_v1.zip
import zipfile, hashlib
zname = [f for f in up if f.endswith(".zip")][0]
zipfile.ZipFile(zname).extractall(".")
print("Extracted", zname)
digest = hashlib.sha256(open(zname, "rb").read()).hexdigest()
expected = "9dcf77389d176661e68308e0c069acac4e0e30f1e33f1cc85f54bc94d313f06d"
assert digest == expected, f"BUNDLE MISMATCH: {digest} != {expected} -- wrong or corrupted zip"
print("sha256 verified:", digest)

In [ ]:
%pip -q install opacus dp-accounting pyyaml scipy optuna ucimlrepo
import torch
print("CUDA:", torch.cuda.is_available(),
      torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU ONLY -- ABORT: runtime will be ~10-20x slower; Runtime > Change runtime type > T4 GPU")

In [ ]:
# --- PRE-FLIGHT 1/2: run the bundled estimator test suite (aborts) -----------
import subprocess, sys
r = subprocess.run([sys.executable, "-m", "unittest", "tests.test_empirical", "-v"],
                   capture_output=True, text=True)
print(r.stderr[-3000:])
assert r.returncode == 0, "TEST SUITE FAILED -- do NOT run E4."
print("Bundled estimator suite: PASS")

In [ ]:
# --- PRE-FLIGHT 2/2: fix markers + E4 invariants (aborts) ---------------------
import sys, os, random, statistics, inspect
sys.path.insert(0, "src"); sys.path.insert(0, "codex")
assert os.environ.get("SMOKE", "0") != "1", "SMOKE is set -- unset it for real E4 cells"
from dp_audit_tightness.privacy.empirical import estimate_empirical_lower_bound
from dp_audit_tightness.privacy.pld_accounting import compute_epsilon_pld
import run_raw_lira_pilot as pilot
import run_decomposition_sweep as decomp

checks = []
# (a) holdout estimator is PRIMARY (I1)
sig = inspect.signature(estimate_empirical_lower_bound)
checks.append(("estimator exposes threshold_selection", "threshold_selection" in sig.parameters))
brr = inspect.getsource(pilot.build_result_row)
checks.append(("build_result_row primary = holdout", 'threshold_selection="holdout"' in brr))
checks.append(("in-sample kept as diagnostic only", "epsilon_lower_conservative_insample" in brr))
random.seed(608)
kw = dict(delta=1e-5, align_event_to_score_direction=True, require_member_favoring=True,
          report_confidence_supported_lower_bound=True, score_direction="higher")
m = [random.gauss(0, 1) for _ in range(640)]; n = [random.gauss(0, 1) for _ in range(640)]
e0 = estimate_empirical_lower_bound(member_scores=m, nonmember_scores=n,
                                    threshold_selection="holdout", **kw)
checks.append(("holdout null -> eps=0", e0.epsilon_lower_empirical_conservative == 0.0))
m = [random.gauss(3, 1) for _ in range(640)]; n = [random.gauss(0, 1) for _ in range(640)]
e1 = estimate_empirical_lower_bound(member_scores=m, nonmember_scores=n,
                                    threshold_selection="holdout", **kw)
checks.append(("holdout signal -> eps>0", e1.epsilon_lower_empirical_conservative > 0.5))
# (b) symmetric scorer + OUT-count matching (I4)
rls_sig = inspect.signature(pilot.raw_lira_scores)
checks.append(("match_out_counts=True default", rls_sig.parameters["match_out_counts"].default is True))
src = inspect.getsource(pilot.raw_lira_scores)
checks.append(("SYMMETRIC scorer present", "fmean(out_losses) - float(target_" in src.replace("\n", "")))
checks.append(("disjoint sampler present", hasattr(pilot, "sample_query_indices_disjoint")))
checks.append(("quality flags present", hasattr(pilot, "apply_quality_flags")))
# (c) E4 knobs match plan SE4
checks.append(("K=32 shadows", decomp.K_SHADOWS == 32))
checks.append(("5120 disjoint queries", decomp.QUERY_BUDGET * len(decomp.AUDIT_SEEDS) == 5120))
checks.append(("1000 matched canaries", decomp.NUM_CANARIES == 1000))
checks.append(("canonical seeds 123/124/125", decomp.CANONICAL_TRAINING_SEEDS == [123, 124, 125]))
checks.append(("I5 matched canary strategy", '"matched_random_canaries"' in inspect.getsource(decomp.canary_audit_target)))
checks.append(("I2 hard google gate", "google_dp_accounting" in inspect.getsource(decomp.enforce_accounting_invariant)))
# (d) share formulas telescope to 1 on synthetic rows
def _wide(threat, w, gdp, valid):
    return {"dataset": "x", "target_epsilon": 1.0, "training_seed": 123, "threat_model": threat,
            "epsilon_upper_rdp": 2.0, "epsilon_upper_tighter": 1.0,
            "epsilon_lower_conservative": w, "epsilon_lower_gdp": gdp,
            "gdp_valid_lower_bound": valid, "quality_flag": "ok"}
s = decomp.compute_shares(_wide("passive", 0.1, 0.15, True), _wide("canary", 0.3, 0.4, True))
checks.append(("shares telescope to 1", s["shares_sum"] is not None and abs(s["shares_sum"] - 1.0) < 1e-9))
s2 = decomp.compute_shares(_wide("passive", 0.1, 0.15, True), _wide("canary", 0.3, 1.5, False))
checks.append(("gdp-invalid renormalizes", s2["estimator_share_excluded"] and abs(s2["shares_sum"] - 1.0) < 1e-9))
# (e) hard PLD backend live check (I2)
res = compute_epsilon_pld(noise_multiplier=1.1, sampling_rate=128/2048, num_steps=16, delta=1e-5)
checks.append(("PLD backend is google_dp_accounting", res["backend_used"] == "google_dp_accounting"))

failed = [name for name, ok in checks if not ok]
for name, ok in checks: print(("PASS " if ok else "FAIL ") + name)
if failed:
    raise SystemExit(f"SANITY CHECKS FAILED: {failed} -- do NOT run E4.")
print("\nAll E4 pre-flight checks passed -- safe to run.")

In [ ]:
# --- Stage E1/E2 results (upload hpo_e1e2_results_<dataset>.zip) --------------
from google.colab import files
from pathlib import Path
import zipfile
up = files.upload()
for name in up:
    if name.endswith(".zip"):
        zipfile.ZipFile(name).extractall(".")
        print("extracted", name)
best = sorted(Path("results/hpo/best_configs").glob("*.yaml"))
assert best, "results/hpo/best_configs is empty -- E4 audits tuned models only (run E1 first)"
print(f"{len(best)} tuned configs:")
for p in best:
    print("  ", p.name)
ckpts = list(Path("results/training/checkpoints").glob("*.pt"))
print(f"{len(ckpts)} E2 checkpoints found (targets retrain automatically if missing)")

In [ ]:
# --- PARAMETERS ----------------------------------------------------------------
DATASET = "mnist"     # "mnist" | "diabetes" (diabetes = passive-only cells for now)
EPSILON = 1.0         # one epsilon per session recommended; None = all cells for DATASET
print(DATASET, EPSILON)

In [ ]:
# --- Data prep -------------------------------------------------------------------
import subprocess, sys
from pathlib import Path
if DATASET == "diabetes" and not Path("data/raw/cdc_diabetes.npz").exists():
    r = subprocess.run([sys.executable, "experiments/prepare_diabetes.py", "--fetch-uci"])
    assert r.returncode == 0, "cdc_diabetes.npz build failed"
print("data ready")

In [ ]:
# --- Run the decomposition sweep (resume-friendly; flushes after every row) ------
import subprocess, sys, time
cmd = [sys.executable, "codex/run_decomposition_sweep.py", "--dataset", DATASET]
if EPSILON is not None:
    cmd += ["--epsilon", str(EPSILON)]
print(" ".join(cmd)); started = time.time()
r = subprocess.run(cmd)
assert r.returncode == 0, "decomposition sweep failed"
print(f"done in {(time.time()-started)/60:.1f} min")

In [ ]:
# --- Results + gate G2 (shares must telescope to 1 +/- 0.02) ---------------------
import pandas as pd
rows = pd.read_csv("results/decomposition/rows.csv")
shares = pd.read_csv("results/decomposition/shares.csv")
sel = rows[rows.dataset == DATASET]
if EPSILON is not None:
    sel = sel[sel.target_epsilon == EPSILON]
display(sel[["dataset", "target_epsilon", "training_seed", "threat_model", "estimator",
             "epsilon_lower", "epsilon_upper_rdp", "epsilon_upper_pld", "quality_flag"]])
per_seed = shares[shares.row_kind == "per_seed"]
per_seed = per_seed[per_seed.dataset == DATASET]
if EPSILON is not None:
    per_seed = per_seed[per_seed.target_epsilon == EPSILON]
display(per_seed[["dataset", "target_epsilon", "training_seed", "accounting_share",
                  "threat_share", "estimator_share", "residual_share", "shares_sum",
                  "shares_sum_ok", "estimator_share_excluded", "threat_share_excluded"]])
bad = per_seed[per_seed.shares_sum_ok == False]  # noqa: E712
assert len(bad) == 0, "G2 FAILURE: shares do not telescope -- inspect before running more cells"
print("G2 ok: shares telescope to 1 within 0.02")
agg = shares[shares.row_kind == "cell_aggregate"]
display(agg)

In [ ]:
# --- Download results (decomposition + training, for the next session) ----------
import shutil
from google.colab import files
tag = DATASET if EPSILON is None else f"{DATASET}_eps{EPSILON}"
shutil.make_archive(f"decomposition_results_{tag}", "zip", ".", "results/decomposition")
shutil.make_archive(f"training_results_{tag}", "zip", ".", "results/training")
files.download(f"decomposition_results_{tag}.zip")
files.download(f"training_results_{tag}.zip")